# Iceland Under Pressure — Data Merging Pipeline

**Course:** 02806 Social Data Analysis and Visualization, Spring 2026  
**Authors:** Ylfa Margrét Ólafsdóttir & Tanja Kristín Árnadóttir

This notebook does the data cleaning and merging. It produces three CSV files that the explainer notebook loads:

- `iceland_panel.csv` — yearly panel with all merged indicators
- `iceland_income_by_age.csv` — income broken down by age bucket
- `airbnb_iceland.csv` — trimmed Airbnb listings

The output files are committed to the GitHub repository so the merge can be inspected without running any code.


## Setup and paths

Update `DATA_DIR` and `OUT_DIR` if your folders are in a different location.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.ticker import FuncFormatter
from functools import reduce
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Please change this
DATA_DIR = Path('/Users/tanjakristin/Desktop/Python/s252892.github.io/notebook/Data/')
OUT_DIR = Path('/Users/tanjakristin/Desktop/Python/s252892.github.io/notebook/Data/')
OUT_DIR.mkdir(exist_ok=True)

# Raw input files
PATH_TEK    = DATA_DIR / 'TEK01001_20260429-205548.xlsx'
PATH_MAN10  = DATA_DIR / 'MAN10001_20260429-185904.xlsx'
PATH_LIF210 = DATA_DIR / 'LIF03210_20260429-190103.xlsx'
PATH_LIF340 = DATA_DIR / 'LIF03340_20260429-191006.xlsx'
PATH_VIS    = DATA_DIR / 'VIS01106_20260429-195620.xlsx'
PATH_CPI    = DATA_DIR / 'VIS01005_20260429-195830.xlsx'
PATH_HMS    = DATA_DIR / 'aMmcEWGNHVfTPSjX_Gogn_manadarskyrsla_agust25HMS-1-.xlsx'
PATH_IDN    = DATA_DIR / 'IDN03001_20260429-205330.xlsx'
PATH_AIRBNB = DATA_DIR / 'part-00442-e685bf03-a693-439c-b3bb-b8b063ff0db5.c000.csv'
PATH_POP_LONG = DATA_DIR / 'MAN08000_20260429-212856.xlsx'

# Exported analysis files
PATH_PANEL = OUT_DIR / 'iceland_panel.csv'
PATH_INCOME_AGE = OUT_DIR / 'iceland_income_by_age.csv'
PATH_AIRBNB_TRIMMED = OUT_DIR / 'airbnb_iceland.csv'


## Cleaning and merging

Cleans each raw source into a yearly indicator table and merges them into one panel. Only variables used in the website are kept.


In [2]:
# Clean each source into a yearly indicator table, then merge only the variables used in the website.

raw = pd.read_excel(PATH_TEK, header=None)
years = [int(y) for y in raw.iloc[2, 4:].tolist() if pd.notna(y)]  # was 3:, should be 4:

# Only rows where col 3 has a value (the age label column)
age_rows = raw.iloc[3:].copy()
age_rows = age_rows[age_rows.iloc[:, 3].notna()]

records = []
for _, row in age_rows.iterrows():
    age = str(row.iloc[3]).strip()
    for yr, val in zip(years, row.iloc[4:4+len(years)].tolist()):
        records.append({'year': yr, 'age_group': age,
                        'mean_income_kisk': pd.to_numeric(val, errors='coerce')})
df_inc = pd.DataFrame(records)
df_inc['mean_income_kisk'] = pd.to_numeric(df_inc['mean_income_kisk'], errors='coerce')

# Collapse to 4 broad buckets
bucket_map = {
    '16 - 19 years': '16–29', '20 - 24 years': '16–29', '25 - 29 years': '16–29',
    '30 - 34 years': '30–44', '35 - 39 years': '30–44', '40 - 44 years': '30–44',
    '45 - 49 years': '45–59', '50 - 54 years': '45–59', '55 - 59 years': '45–59',
    '60 - 64 years': '60+',   '65 - 69 years': '60+',   '70 - 74 years': '60+',
    '75 years and older': '60+',
}
df_inc['bucket'] = df_inc['age_group'].map(bucket_map)
income_by_age = (
    df_inc.dropna(subset=['bucket', 'mean_income_kisk'])
    .groupby(['year', 'bucket'])['mean_income_kisk'].mean()
    .reset_index()
)

# Also keep total income (for main panel)
income_total = (
    df_inc[df_inc['age_group'] == 'Total']
    .groupby('year')['mean_income_kisk'].mean()
    .reset_index()
    .rename(columns={'mean_income_kisk': 'mean_income_kisk_total'})
)

print('income_by_age:', income_by_age.shape)
print('Buckets:', income_by_age.bucket.unique())
income_by_age.tail(8)

# ---

raw = pd.read_excel(PATH_MAN10, header=None)
data_rows = raw.iloc[2:].copy()
data_rows.columns = ['quarter', 'group', 'population']
data_rows = data_rows[data_rows['population'].notna() & data_rows['group'].notna()]
data_rows = data_rows[~data_rows['quarter'].astype(str).str.startswith('Population')]
data_rows['quarter'] = data_rows['quarter'].ffill()
data_rows['population'] = pd.to_numeric(data_rows['population'], errors='coerce')
data_rows['q'] = data_rows['quarter'].astype(str).str[-2:]
data_rows['year'] = data_rows['quarter'].astype(str).str[:4].astype(int)

population_annual = (
    data_rows[(data_rows['group'] == 'Total') & (data_rows['q'] == 'Q4')]
    [['year', 'population']].reset_index(drop=True)
)
print(population_annual.shape)
population_annual.tail(5)

# ---

raw = pd.read_excel(PATH_LIF210, header=None)
years = [int(y) for y in raw.iloc[4, 2:].tolist() if pd.notna(y)]
data_rows = raw.iloc[5:].copy()
data_rows = data_rows[data_rows.iloc[:, 0].notna() | data_rows.iloc[:, 1].notna()]
data_rows = data_rows[data_rows.iloc[:, 0].astype(str).str.len() < 100]

records = []
for _, row in data_rows.iterrows():
    tenure = str(row.iloc[0]).strip() if pd.notna(row.iloc[0]) else np.nan
    group  = str(row.iloc[1]).strip() if pd.notna(row.iloc[1]) else np.nan
    for yr, val in zip(years, row.iloc[2:2+len(years)].tolist()):
        records.append({'year': yr, 'tenure_status': tenure, 'group': group, 'proportion_pct': val})

tenure_annual = (
    pd.DataFrame(records)
    .query("tenure_status.str.contains('Owners', na=False) and group == 'Total'")
    [['year', 'proportion_pct']]
    .rename(columns={'proportion_pct': 'owner_occupier_pct'})
    .reset_index(drop=True)
)
print(tenure_annual.shape)
tenure_annual.tail(5)

# ---

raw = pd.read_excel(PATH_LIF340, header=None)
data_rows = raw.iloc[2:].copy()
data_rows.columns = ['year', 'condition', 'sex', 'age_group', 'n_individuals']
data_rows = data_rows[data_rows['n_individuals'].notna() & data_rows['condition'].notna()]
data_rows['year'] = pd.to_numeric(data_rows['year'].ffill(), errors='coerce')
data_rows['n_individuals'] = pd.to_numeric(data_rows['n_individuals'], errors='coerce')
data_rows = data_rows[data_rows['year'].notna()]
data_rows['year'] = data_rows['year'].astype(int)

housing_cond_annual = (
    data_rows[
        data_rows['condition'].str.contains('Poor', na=False) &
        (data_rows['sex'] == 'Total') &
        (data_rows['age_group'] == 'Total')
    ]
    [['year', 'n_individuals']]
    .rename(columns={'n_individuals': 'n_poor_housing'})
    .reset_index(drop=True)
)
print(housing_cond_annual.shape)
housing_cond_annual.tail(5)

# ---

# Price index
raw = pd.read_excel(PATH_VIS, header=None)
raw.columns = ['date', 'price_index']
raw = raw.iloc[2:].copy()
raw = raw[raw['date'].astype(str).str.match(r'\d{4}M\d{2}')]
raw['date'] = pd.to_datetime(raw['date'].astype(str).str.replace('M', '-'), format='%Y-%m')
raw['price_index'] = pd.to_numeric(raw['price_index'], errors='coerce')
raw['year'] = raw['date'].dt.year
price_index_annual = (
    raw.groupby('year')['price_index'].mean().reset_index()
    .rename(columns={'price_index': 'residential_price_index'})
)

# CPI
raw_cpi = pd.read_excel(PATH_CPI, header=None).iloc[4:].copy()
raw_cpi.columns = ['year', 'cpi', 'cpi_ex_housing']
raw_cpi = raw_cpi[pd.to_numeric(raw_cpi['year'], errors='coerce').notna()]
raw_cpi['year'] = raw_cpi['year'].astype(int)
raw_cpi['cpi'] = pd.to_numeric(raw_cpi['cpi'], errors='coerce')
cpi_annual = raw_cpi[['year', 'cpi']].dropna().reset_index(drop=True)

print('price_index:', price_index_annual.shape, '| cpi:', cpi_annual.shape)
price_index_annual.tail(3)

# ---

hms = pd.read_excel(PATH_HMS, sheet_name=None, header=None)

def hms_to_df(sheet, cols):
    raw = hms[sheet].copy()
    for i, row in raw.iterrows():
        v = str(row.iloc[0])
        if v not in ('nan','NaT') and not v.startswith(('Dags','Unnamed','Mánaðar')):
            start = i
            break
    df = raw.iloc[start:, :len(cols)].copy()
    df.columns = cols
    return df.dropna(subset=[cols[0]]).reset_index(drop=True)

# LM.1: loan burden
# We keep loan burden because it supports the affordability part of the story.
lm1 = hms_to_df('LM.1', ['date', 'loan_type', 'payment_per_10m'])
lm1['date'] = pd.to_datetime(lm1['date'], errors='coerce')
lm1['payment_per_10m'] = pd.to_numeric(lm1['payment_per_10m'], errors='coerce')
lm1['year'] = lm1['date'].dt.year

lm1_annual = (
    lm1.dropna(subset=['date'])
    .groupby('year')['payment_per_10m']
    .mean()
    .reset_index()
    .rename(columns={'payment_per_10m': 'avg_loan_payment_per_10m_isk'})
)

print('LM.1 loan burden:', lm1_annual.shape)
lm1_annual.tail(5)

# ---

raw = pd.read_excel(PATH_IDN, header=None).iloc[4:].copy()
raw.columns = ['year', 'dwellings_begun', 'cubic_meters']
raw = raw[pd.to_numeric(raw['year'], errors='coerce').notna()]
raw['year'] = raw['year'].astype(int)
raw['dwellings_begun'] = pd.to_numeric(raw['dwellings_begun'], errors='coerce')
dwellings_annual = raw[['year', 'dwellings_begun']].dropna().reset_index(drop=True)
print(dwellings_annual.shape)
dwellings_annual.tail(5)

# ---

bm2 = hms['BM.2'].copy()
bm2.columns = ['year', 'pop_growth_rate', 'dwelling_growth_rate']
bm2 = bm2.iloc[1:].copy()
bm2['year'] = pd.to_numeric(bm2['year'], errors='coerce')
bm2['pop_growth_rate'] = pd.to_numeric(bm2['pop_growth_rate'], errors='coerce')
bm2['dwelling_growth_rate'] = pd.to_numeric(bm2['dwelling_growth_rate'], errors='coerce')
bm2_annual = bm2.dropna().reset_index(drop=True)
bm2_annual['year'] = bm2_annual['year'].astype(int)
print(bm2_annual.shape)
bm2_annual.tail(5)

# ---

from functools import reduce

annual_dfs = [
    income_total,
    population_annual,
    tenure_annual,
    housing_cond_annual,
    price_index_annual,
    cpi_annual,
    lm1_annual,
    dwellings_annual,
    bm2_annual,
]

panel = reduce(lambda a, b: pd.merge(a, b, on='year', how='outer'), annual_dfs)
panel = panel.sort_values('year').reset_index(drop=True)

# Derived columns
cpi_base = panel.loc[panel.year==2000, 'cpi'].values[0]
inc_base = panel.loc[panel.year==2000, 'mean_income_kisk_total'].values[0]
pi_base  = panel.loc[panel.year==2000, 'residential_price_index'].values[0]

panel['income_idx'] = panel['mean_income_kisk_total'] / inc_base * 100
panel['price_idx']  = panel['residential_price_index'] / pi_base * 100
panel['real_price_index'] = panel['residential_price_index'] / (panel['cpi'] / cpi_base)
panel['monthly_income_isk'] = panel['mean_income_kisk_total'] * 1000 / 12
panel['loan_burden_pct_income'] = panel['avg_loan_payment_per_10m_isk'] / panel['monthly_income_isk'] * 100
panel['poor_housing_pct'] = panel['n_poor_housing'] / panel['population'] * 100

# Cumulative housing deficit (from 2006 where BM.2 starts)
bm2_sub = panel[panel.year.between(2006,2026)].copy()
bm2_sub['gap'] = bm2_sub['pop_growth_rate'] - bm2_sub['dwelling_growth_rate']
bm2_sub['cumulative_gap'] = bm2_sub['gap'].cumsum()
panel = panel.merge(bm2_sub[['year','gap','cumulative_gap']], on='year', how='left')

print(f'Panel: {panel.shape[0]} years × {panel.shape[1]} columns')
print(f'Year range: {panel.year.min()} – {panel.year.max()}')
panel.columns.tolist()

income_by_age: (140, 3)
Buckets: <StringArray>
['16–29', '30–44', '45–59', '60+']
Length: 4, dtype: str
(16, 2)
(13, 2)
(15, 2)
price_index: (27, 2) | cpi: (38, 2)
LM.1 loan burden: (22, 2)
(52, 2)
(22, 3)
Panel: 58 years × 19 columns
Year range: 1970 – 2027


['year',
 'mean_income_kisk_total',
 'population',
 'owner_occupier_pct',
 'n_poor_housing',
 'residential_price_index',
 'cpi',
 'avg_loan_payment_per_10m_isk',
 'dwellings_begun',
 'pop_growth_rate',
 'dwelling_growth_rate',
 'income_idx',
 'price_idx',
 'real_price_index',
 'monthly_income_isk',
 'loan_burden_pct_income',
 'poor_housing_pct',
 'gap',
 'cumulative_gap']

## Save outputs

The cleaned panel, the income-by-age table, and the trimmed Airbnb file are saved as CSVs. These are what the explainer notebook loads.


In [3]:
# Main panel
panel.to_csv(PATH_PANEL, index=False)
print('Saved iceland_panel.csv')

# Income by age — separate file (used for interactive chart)
income_by_age.to_csv(PATH_INCOME_AGE, index=False)
print('Saved iceland_income_by_age.csv')

# Airbnb — load and save trimmed version
df_airbnb = pd.read_csv(PATH_AIRBNB, low_memory=False, on_bad_lines='skip')
keep = ['listing_id','listing_type','room_type','latitude','longitude',
        'guests','bedrooms','superhost','num_reviews','rating_overall',
        'ttm_revenue_native','ttm_avg_rate_native','ttm_occupancy',
        'ttm_reserved_days','ttm_revpar_native','currency']
df_airbnb[[c for c in keep if c in df_airbnb.columns]].to_csv(
    PATH_AIRBNB_TRIMMED, index=False)
print('Saved airbnb_iceland.csv')

Saved iceland_panel.csv
Saved iceland_income_by_age.csv
Saved airbnb_iceland.csv
